# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniyalhaider236/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*



Same March 2026 slice and Lane 2 framing as `w03_data_contract.ipynb`, extended with the two things that
notebook didn't need yet: real categorical handling (`content_type` one-hot) and explicit, documented
missing-value fills instead of silent zeros.

In [6]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os
import pandas as pd
import numpy as np
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. In Colab open the 🔑 Secrets panel, add a secret "
        "named HF_TOKEN, paste your Hugging Face READ token as its value, and enable notebook access."
    )

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected to the FlyRank warehouse.")

# Same feature-frame query as w03_data_contract.ipynb (proven to run against this warehouse)
feature_frame = con.sql(f"""
WITH march AS (
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
),
agg AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date < DATE '2026-03-16' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_prev_half,
        SUM(CASE WHEN report_date < DATE '2026-03-16' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_prev_half,
        AVG(CASE WHEN report_date < DATE '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev_half,
        COUNT(DISTINCT CASE WHEN report_date < DATE '2026-03-16' AND COALESCE(gsc_impressions, 0) > 0 THEN report_date END) AS days_with_impressions_prev_half,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_later_half
    FROM march GROUP BY 1, 2
),
content_meta AS (
    SELECT content_hash_id, ANY_VALUE(content_created_date) AS content_created_date, ANY_VALUE(content_type) AS content_type
    FROM {TABLES["dim_content"]} GROUP BY content_hash_id
)
SELECT
    a.client_hash_id, a.content_hash_id,
    a.impressions_prev_half, a.clicks_prev_half, a.avg_position_prev_half, a.days_with_impressions_prev_half,
    DATE_DIFF('day', m.content_created_date, DATE '2026-03-01') AS content_age_days,
    m.content_type,
    a.impressions_later_half,
    CASE WHEN a.impressions_prev_half > 0 AND a.impressions_later_half < 0.8 * a.impressions_prev_half
         THEN 1 ELSE 0 END AS is_declining_proxy
FROM agg a
LEFT JOIN content_meta m USING (content_hash_id)
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")

# --- Categorical handling: one-hot the content type ---
feature_frame["content_type"] = feature_frame["content_type"].fillna("unknown")
content_type_dummies = pd.get_dummies(feature_frame["content_type"], prefix="ctype")
feature_frame = pd.concat([feature_frame, content_type_dummies], axis=1)
print("\nContent type categories found:", feature_frame["content_type"].unique().tolist())

# --- Documented missing-value fills (not silent zeros) ---
numeric_fill_cols = ["avg_position_prev_half", "content_age_days"]
for c in numeric_fill_cols:
    n_missing = feature_frame[c].isna().sum()
    feature_frame[c + "_missing"] = feature_frame[c].isna().astype(int)
    fill_value = feature_frame[c].median()
    feature_frame[c] = feature_frame[c].fillna(fill_value)
    print(f"{c}: {n_missing} missing ({n_missing/len(feature_frame):.1%}) — filled with median ({fill_value:.1f}), flagged in {c}_missing")

display(feature_frame.head(10))


Connected to the FlyRank warehouse.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 331,437

Content type categories found: ['keyword article', 'feedly article', 'comparison article']
avg_position_prev_half: 180762 missing (54.5%) — filled with median (8.8), flagged in avg_position_prev_half_missing
content_age_days: 0 missing (0.0%) — filled with median (190.0), flagged in content_age_days_missing


,client_hash_id,content_hash_id,impressions_prev_half,clicks_prev_half,avg_position_prev_half,days_with_impressions_prev_half,content_age_days,content_type,impressions_later_half,is_declining_proxy,ctype_comparison article,ctype_feedly article,ctype_keyword article,avg_position_prev_half_missing,content_age_days_missing
0,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
1,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
2,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0.0,0.0,8.756892,0,145,keyword article,1.0,0,False,False,True,1,0
3,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
4,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,0.0,8.756892,0,154,keyword article,0.0,0,False,False,True,1,0
5,client_0797ff3a1fc9a6a5,content_044c54ec4adcc4b2,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
6,client_0797ff3a1fc9a6a5,content_07573a1cc2034981,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
7,client_0797ff3a1fc9a6a5,content_084680e7da2a2ff9,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
8,client_0797ff3a1fc9a6a5,content_0c3410828632f110,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0
9,client_0797ff3a1fc9a6a5,content_0e62212f207dde7a,0.0,0.0,8.756892,0,145,keyword article,0.0,0,False,False,True,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Categorical? | Available before decision? |
|---|---|---|---|---|
| `impressions_prev_half` | Search impressions, first half of March | Query-level `COALESCE(...,0)` — absence in the daily fact means zero activity that day, a real zero, not unknown | No | Yes — historical |
| `clicks_prev_half` | Search clicks, first half of March | Same as above — real zero, not missing | No | Yes — historical |
| `avg_position_prev_half` | Mean search position, first half of March | True missing (no impressions that half) → filled with the column median, flagged in `avg_position_prev_half_missing` | No | Yes — historical |
| `days_with_impressions_prev_half` | Count of distinct days with any impressions | Never missing (a `COUNT`, floors at 0) | No | Yes — historical |
| `content_age_days` | Days between content creation and the decision date | True missing (no creation date on record) → filled with median, flagged in `content_age_days_missing` | No | Yes — metadata, fixed at creation |
| `content_type` (one-hot: `ctype_*`) | Article format/category | Missing category filled as `"unknown"` before one-hot, so it gets its own honest column rather than silently vanishing | Yes | Yes — metadata, fixed at creation |

**Outcome, not a feature:** `impressions_later_half` and `is_declining_proxy` exist in this frame
deliberately, for the leakage experiment in Section 3 — never as model inputs.

In [7]:
print("Feature columns actually usable as model inputs (excludes IDs and outcome columns):")
usable = [c for c in feature_frame.columns if c not in
          ["client_hash_id", "content_hash_id", "content_type", "impressions_later_half", "is_declining_proxy"]]
print(usable)
print(f"\nTotal usable features: {len(usable)}")


Feature columns actually usable as model inputs (excludes IDs and outcome columns):
['impressions_prev_half', 'clicks_prev_half', 'avg_position_prev_half', 'days_with_impressions_prev_half', 'content_age_days', 'ctype_comparison article', 'ctype_feedly article', 'ctype_keyword article', 'avg_position_prev_half_missing', 'content_age_days_missing']

Total usable features: 10


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*



Two attacks on my own feature set: first a blind scan of the *entire* warehouse schema for columns that
sound label-derived or future-facing (not just the handful I already know about), then the same
honest-vs-leaky quantitative test from `w03_data_contract.ipynb`, re-run here on the extended feature set.

In [8]:
# --- Attack 1: scan the FULL schema (not a filtered subset) for suspicious column names ---
suspect_words = ["trend", "flag", "health", "optimi", "declin", "later", "next_", "future", "outcome", "score"]

for table_name in ["fact_daily", "dim_content", "dim_clients"]:
    full_schema = con.sql(f'DESCRIBE SELECT * FROM {TABLES[table_name]}').df()
    hits = full_schema[full_schema["column_name"].str.lower().str.contains("|".join(suspect_words))]
    print(f"=== {table_name}: {len(full_schema)} total columns ===")
    if len(hits):
        print("Suspicious column names found:")
        print(hits[["column_name", "column_type"]].to_string(index=False))
    else:
        print("No column names matched the suspect-word list.")
    print()

# --- Attack 2: quantify the known leak the same way w03_data_contract.ipynb did ---
from sklearn.tree import DecisionTreeClassifier

honest_features = usable
model_df = feature_frame.dropna(subset=honest_features + ["is_declining_proxy"]).copy()

X_honest = model_df[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["is_declining_proxy"].astype(int)
honest_model = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
honest_model.fit(X_honest, y)
honest_accuracy = honest_model.score(X_honest, y)

leaky_features = honest_features + ["impressions_later_half"]
X_leaky = model_df[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky_model = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
leaky_model.fit(X_leaky, y)
leaky_accuracy = leaky_model.score(X_leaky, y)

print(f"Honest feature-set training accuracy: {honest_accuracy:.3f}")
print(f"Leaky feature-set training accuracy:  {leaky_accuracy:.3f}")
print(f"Change caused by leakage:              {leaky_accuracy - honest_accuracy:+.3f}")

=== fact_daily: 31 total columns ===
No column names matched the suspect-word list.

=== dim_content: 26 total columns ===
Suspicious column names found:
               column_name column_type
       last_optimized_date        DATE
optimization_eligible_date        DATE

=== dim_clients: 9 total columns ===
No column names matched the suspect-word list.

Honest feature-set training accuracy: 0.705
Leaky feature-set training accuracy:  0.691
Change caused by leakage:              -0.014


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`impressions_later_half`** — the outcome window itself; deliberately proven leaky above, then removed.
- **`is_declining_proxy`** — the label. Never a feature by definition.
- **Any column matched by the schema scan in Section 3** (trend/flag/health/score-named columns, if
  present in this warehouse release) — excluded on the same principle established in Week 5/6: even a
  partial or indirect overlap with how the label is built disqualifies a column, regardless of how strong
  its individual lift looks.
- **`client_hash_id` / `content_hash_id`** — used only for grouping and joins, never as predictive
  features; a model shouldn't learn "which specific client/page this is."
- **Private client names, real URLs, or raw queries** — never pulled into this or any notebook; this
  warehouse only exposes anonymized hash IDs.

In [9]:
print("Final excluded-from-modeling list:")
print(["impressions_later_half", "is_declining_proxy", "client_hash_id", "content_hash_id", "content_type"])
print("\n(content_type itself is excluded as a raw string — its one-hot ctype_* columns are the usable form)")


Final excluded-from-modeling list:
['impressions_later_half', 'is_declining_proxy', 'client_hash_id', 'content_hash_id', 'content_type']

(content_type itself is excluded as a raw string — its one-hot ctype_* columns are the usable form)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.